# Subtitle Cleaner and Translator

## 1. Project Overview

**Task:** Parse, clean, and translate subtitle files (SRT format) — removing timestamps, formatting tags, and annotations, then translating the clean text while preserving timing.

**Stack:** `LangChain` + `ChatOllama` + `qwen3.5:9b` for translation, regex for SRT parsing.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Parse** SRT subtitle format into structured data |
| 2 | **Clean** subtitle text — remove tags, annotations, formatting |
| 3 | **Translate** subtitle segments while preserving timing |
| 4 | **Reconstruct** valid SRT output in the target language |

## 3. Why This Matters

- **Video content** is consumed globally; subtitles need translation
- **Raw SRT files** contain noise (tags, formatting, annotations)
- **Timing preservation** is critical — translations must fit the original segments

## 4. Setup

In [ ]:
import os, json, re, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

LLM_MODEL = "qwen3.5:9b"
llm = ChatOllama(model=LLM_MODEL, temperature=0.2)

def clean(text):
    if "<think>" in text: text = text.split("</think>")[-1].strip()
    return text.strip()

def ask(system, user):
    return clean(llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content)

print(f"LLM ready: {ask('Reply with one word.', 'Say ready.')[:20]}")

## 5. Sample SRT Content

In [ ]:
SRT_CONTENT = """1
00:00:01,000 --> 00:00:03,500
Hello and welcome to this tutorial.

2
00:00:03,600 --> 00:00:06,000
Today we will learn about machine learning.

3
00:00:06,100 --> 00:00:09,000
<i>It is one of the most important fields</i>
in computer science.

4
00:00:09,100 --> 00:00:12,000
[Music] Let's get started!

5
00:00:12,100 --> 00:00:15,000
First, we need to understand {\\an8}what data is.

6
00:00:15,100 --> 00:00:18,500
Data can be structured or unstructured.
"""

print("RAW SRT CONTENT:")
print("=" * 50)
print(SRT_CONTENT)

---
## 6. SRT Parser

In [ ]:
def parse_srt(srt_text):
    pattern = re.compile(
        r"(\d+)\s*\n"
        r"(\d{2}:\d{2}:\d{2},\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2},\d{3})\s*\n"
        r"((?:.+\n?)*)",
        re.MULTILINE
    )
    segments = []
    for match in pattern.finditer(srt_text):
        idx = int(match.group(1))
        start = match.group(2)
        end = match.group(3)
        text = match.group(4).strip()
        segments.append({"index": idx, "start": start, "end": end, "text": text})
    return segments

segments = parse_srt(SRT_CONTENT)
print(f"Parsed {len(segments)} subtitle segments:")
for seg in segments:
    print(f"  [{seg['index']}] {seg['start']} -> {seg['end']}: {seg['text'][:60]}")

## 7. Text Cleaning

In [ ]:
def clean_subtitle_text(text):
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\{\\[^}]+\}", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("CLEANED SEGMENTS:")
print("=" * 50)
for seg in segments:
    original = seg["text"]
    cleaned = clean_subtitle_text(original)
    seg["clean_text"] = cleaned
    if original != cleaned:
        print(f"  [{seg['index']}] BEFORE: {original}")
        print(f"  [{seg['index']}] AFTER:  {cleaned}")
    else:
        print(f"  [{seg['index']}] {cleaned}")

full_text = " ".join(seg["clean_text"] for seg in segments)
print(f"\nFull cleaned text ({len(full_text.split())} words):")
print(f"  {full_text}")

## 8. Segment-by-Segment Translation

In [ ]:
TRANSLATE_SYSTEM = "Translate to {language}. Keep it concise (similar word count). Return ONLY the translation. /no_think"

target_lang = "Spanish"
print(f"TRANSLATION: English -> {target_lang}")
print("=" * 70)
for seg in segments:
    translation = ask(TRANSLATE_SYSTEM.format(language=target_lang), seg["clean_text"])
    seg["translated"] = translation
    print(f"  [{seg['index']}] EN: {seg['clean_text']}")
    print(f"  [{seg['index']}] ES: {translation}")
    print()

## 9. Reconstruct Translated SRT

In [ ]:
def build_srt(segments, text_key="translated"):
    lines = []
    for seg in segments:
        lines.append(str(seg["index"]))
        lines.append(f"{seg['start']} --> {seg['end']}")
        lines.append(seg.get(text_key, seg["clean_text"]))
        lines.append("")
    return "\n".join(lines)

translated_srt = build_srt(segments, "translated")
print("TRANSLATED SRT OUTPUT:")
print("=" * 50)
print(translated_srt)

## 10. Multi-Language Translation

In [ ]:
LANGUAGES = ["French", "German", "Portuguese"]
seg = segments[1]
print("MULTI-LANGUAGE COMPARISON")
print("=" * 60)
print(f"  EN: {seg['clean_text']}")
for lang in LANGUAGES:
    trans = ask(TRANSLATE_SYSTEM.format(language=lang), seg["clean_text"])
    print(f"  {lang[:2].upper()}: {trans}")

## 11. Quality Check

In [ ]:
EVAL_SYSTEM = """Compare subtitle translation. Score 1-5:
- accuracy: meaning preserved?
- length_match: similar word count to original?
- naturalness: sounds natural?
Return JSON: {"accuracy":N,"length_match":N,"naturalness":N,"total":N} /no_think"""

print("TRANSLATION QUALITY")
print("=" * 50)
for seg in segments[:3]:
    resp = ask(EVAL_SYSTEM, f"Original: {seg['clean_text']}\nTranslation: {seg.get('translated', 'N/A')}")
    text = resp
    if "```" in text: text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "")
    s, e = text.find("{"), text.rfind("}") + 1
    if s >= 0 and e > s:
        try:
            sc = json.loads(text[s:e])
            total = sum(sc.get(k,0) for k in ["accuracy","length_match","naturalness"])
            print(f"  [{seg['index']}] {total}/15")
        except: pass

## 12. Common Mistakes & Key Takeaways

| Mistake | Fix |
|---------|-----|
| Translating raw SRT | Clean tags/timestamps first |
| Ignoring timing | Keep translations concise to fit segments |
| Full-text translation | Translate segment by segment for timing |
| Losing formatting | Preserve SRT structure in output |

| # | Takeaway |
|---|----------|
| 1 | **Parse SRT into segments** before processing |
| 2 | **Clean formatting tags** ([Music], <i>, ASS tags) |
| 3 | **Translate per-segment** to preserve timing alignment |
| 4 | **Length control** ensures text fits the time window |
| 5 | **Reconstruct valid SRT** for immediate use |

---
*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*
